# Harry Potter GPT — Colab runner

This notebook is just a thin driver. All the actual pipeline logic (resume-from-Drive, skip-if-already-done, size validation, backup-after-each-stage) lives in `run_pipeline.py`, `convert_to_hf.py`, and `showcase.py` in the repo — not copy-pasted across notebook cells. If something needs fixing, it gets fixed in one place.

Runtime: Runtime -> Change runtime type -> T4 GPU.

Run the setup cells once, then the pipeline cell. Safe to re-run after any disconnect — completed stages are auto-skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/harry-potter-gpt'

In [ ]:
!git clone https://github.com/abhijitdalal26/harry-potter-gpt.git /content/harry-potter-gpt
%cd /content/harry-potter-gpt/nanoGPT
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q tiktoken transformers datasets trl accelerate

## Run the pipeline
`--stage all` runs everything in order (pretrain -> SFT -> convert -> DPO -> showcase), auto-skipping any stage already backed up in Drive.

To rerun a single stage manually instead, change `--stage all` to one of: `pretrain`, `sft`, `convert`, `dpo`, `showcase`.

In [ ]:
!python run_pipeline.py --drive-backup "{DRIVE_BACKUP}" --stage all

## Read the results
The transcript below is what turns into the real conversation examples on the portfolio page.

In [ ]:
from IPython.display import Markdown, display
with open('results/stage_comparison.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

## Done

Everything of value lives in Google Drive under `harry-potter-gpt/`:
- `out-harry-potter.zip` — Stage 1 nanoGPT checkpoint (pretrain-only)
- `out-harry-potter-sft.zip` — Stage 2 nanoGPT checkpoint (SFT, chat format)
- `harry-potter-hf.zip` — Stage 3, SFT model converted to HuggingFace format
- `harry-potter-hf-dpo.zip` — Stage 4 DPO model (HuggingFace format)
- `stage_comparison.md` — the basic-to-advanced transcript across all three stages
- `harry-potter-gpt-full.zip` — all of the above bundled together

If you want to host a live demo, `harry-potter-hf-dpo.zip` (~474MB) is what gets uploaded to Hugging Face.